# Polynomial features in logistic regression

In [1]:
import warnings
import numpy as np
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
warnings.simplefilter("ignore")

X, y = make_moons(n_samples=200, noise=0.25, random_state=0)     # two interleaved half-moons (the figure's data)
print(X.shape, np.bincount(y))

(200, 2) [100 100]


## Degree 2 turns two columns into five

In [2]:
poly = PolynomialFeatures(degree=2, include_bias=False)
print(poly.fit(X).get_feature_names_out(["x1", "x2"]))
for d in [1, 2, 3, 4, 10, 25]:
    print(d, PolynomialFeatures(degree=d, include_bias=False).fit(X).n_output_features_)

['x1' 'x2' 'x1^2' 'x1 x2' 'x2^2']
1 2
2 5
3 9
4 14
10 65
25 350


## Training and test accuracy for several degrees
20 training sets of 200 points (seeds 0-19), each scored on one large fresh test set of 5,000 points from the same generator. Weak penalty, C = 10,000. Averages over the 20 seeds.

In [3]:
X_test, y_test = make_moons(n_samples=5000, noise=0.25, random_state=999)

def scores(d, C):
    tr, te = [], []
    for s in range(20):
        Xs, ys = make_moons(n_samples=200, noise=0.25, random_state=s)
        m = make_pipeline(PolynomialFeatures(degree=d, include_bias=False), StandardScaler(),
                          LogisticRegression(C=C, max_iter=100000)).fit(Xs, ys)
        tr.append(m.score(Xs, ys)); te.append(m.score(X_test, y_test))
    return np.mean(tr), np.mean(te)

for d in [1, 2, 3, 4, 10, 25]:
    tr, te = scores(d, 1e4)
    print(f"degree {d:2d}: train {tr:.3f}, test {te:.3f}, gap {tr - te:.3f}")

degree  1: train 0.855, test 0.854, gap 0.001


degree  2: train 0.859, test 0.855, gap 0.005


degree  3: train 0.947, test 0.935, gap 0.012


degree  4: train 0.947, test 0.928, gap 0.020


degree 10: train 0.962, test 0.919, gap 0.043


degree 25: train 0.962, test 0.917, gap 0.045


## The penalty strength C (Extra in the Note)

In [4]:
for C in [1e4, 100, 10, 1]:
    print(f"C={C:g}: " + ", ".join(f"degree {d}: train {tr:.3f} test {te:.3f}" for d in [3, 10, 25] for tr, te in [scores(d, C)]))

C=10000: degree 3: train 0.947 test 0.935, degree 10: train 0.962 test 0.919, degree 25: train 0.962 test 0.917


C=100: degree 3: train 0.946 test 0.938, degree 10: train 0.952 test 0.932, degree 25: train 0.952 test 0.930


C=10: degree 3: train 0.933 test 0.932, degree 10: train 0.945 test 0.933, degree 25: train 0.945 test 0.931


C=1: degree 3: train 0.903 test 0.899, degree 10: train 0.922 test 0.915, degree 25: train 0.922 test 0.914


## The scaler: solver steps at degree 10 (figure data, seed 0)

In [5]:
from sklearn.exceptions import ConvergenceWarning
for d in [3, 10]:
    for scale in [True, False]:
        steps = [PolynomialFeatures(degree=d, include_bias=False)] + ([StandardScaler()] if scale else []) + [LogisticRegression(C=1e4, max_iter=1000)]
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always", ConvergenceWarning)
            m = make_pipeline(*steps).fit(X, y)
        print(f"degree {d:2d}, scaler {scale!s:5}: iterations {m[-1].n_iter_[0]:4d}, converged {not any(issubclass(x.category, ConvergenceWarning) for x in w)}, train {m.score(X, y):.3f}")

degree  3, scaler True : iterations   52, converged True, train 0.960
degree  3, scaler False: iterations   44, converged True, train 0.960


degree 10, scaler True : iterations  182, converged True, train 0.955


degree 10, scaler False: iterations  279, converged True, train 0.950
